# 03 - EELS-Auswertung Praktikumsdatensatz

Gleiches Vorgehen wie Notebook 01, aber an einem anderen Datensatz: einer
Lamelle, deren Spektren von 138 bis 650 eV reichen. Messbar sind dort die Kanten
von Si, S, C, Ca und O - die Ca-L2,3-Kante bei 346 eV ist am deutlichsten, und zu
diesem Datensatz gehoeren sechs Ca-Referenzspektren fuer die Feinstrukturanalyse.

**Dieser Datensatz ist nicht im Teilnehmenden-ZIP enthalten** (dort ist nur
`nanopore`). Notebook 03 ist fuer Betreuende gedacht.

**Arbeite Notebook 01 zuerst durch** - dort sind die einzelnen Schritte erklaert,
hier stehen nur noch die Besonderheiten.

In [ ]:
# Interaktive Plots (zoomen, Spektrum je Bildpunkt anklicken).
# Falls die Plots weiss bleiben oder gar nichts erscheint:
# diese Zeile durch  %matplotlib inline  ersetzen und den Kernel neu starten.
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

import hyperspy.api as hs
import exspy  # muss importiert sein, sonst kennt HyperSpy die EELS-/EDX-Signaltypen nicht

# Findet die Messdaten unabhaengig vom Betriebssystem (siehe workshop_data.py)
from workshop_data import load, load_standards

print("HyperSpy", hs.__version__, "| exspy", exspy.__version__)

## 1. Daten laden

In [ ]:
signal = load("praktikum_eels_highloss", signal_type="EELS")
ll = load("praktikum_eels_lowloss", signal_type="EELS")

signal.plot()

In [ ]:
load("praktikum_adf").plot()

## 2. Zero-Loss-Peak ausrichten

In [ ]:
ll.align_zero_loss_peak(also_align=[signal], signal_range=(-10.0, 10.0))

## 3. Modell aufbauen

Sechs Elemente statt drei - entsprechend mehr Kanten, entsprechend laengerer Fit.
Deshalb hier `rebin` mit **8x8** statt 2x2. Das ist eine bewusste Abwaegung:
Ortsaufloesung gegen Rechenzeit. Wenn du Zeit hast, probiere ruhig 4x4.

Achte auf ueberlappende Kanten (Ga-L bei ~1115 eV, W-N, Pt-N...). Wenn der Fit
unruhig wird, ist meist eine Kante schuld, die im gemessenen Energiebereich gar
nicht vollstaendig enthalten ist.

In [ ]:
# Ga, W und Pt standen urspruenglich mit in der Liste. Ihre naechsten Kanten
# liegen bei 1115, 1809 und 2122 eV - gemessen wurde aber nur bis 650 eV.
# exspy hat sie deshalb kommentarlos verworfen. Hier stehen nur noch Elemente,
# deren Kanten im Bereich 138-650 eV tatsaechlich liegen.
signal.add_elements(["Si", "S", "C", "Ca", "O"])

signal_binned = signal.rebin(scale=[8, 8, 1])

# Das Untergrundfenster muss VOR der ersten Kante liegen. Erste Kante ist
# Si-L1 bei 150 eV, die Messung beginnt bei 138 eV - viel Platz ist nicht.
# (Urspruenglich stand hier (70, 96), aus Notebook 01 kopiert. Dieser Bereich
# liegt komplett ausserhalb dieser Daten - ohne dass es eine Fehlermeldung gab.)
signal_binned = signal_binned.remove_background(signal_range=(138.5, 148.0))

m = signal_binned.create_model(auto_background=False)
m.components

In [ ]:
# --- Variante A: interaktiv ---
m.gui()

In [ ]:
# --- Variante B: per Code ---
for komponente in m:
    print(f"{komponente.name}   aktiv={komponente.active}")

In [ ]:
m.plot()

In [ ]:
m.multifit(kind="smart")

In [ ]:
m.plot_results()

## 4. Feinstruktur der Ca-L2,3-Kante

**Abweichung vom Original:** Dort wurden die Si-Referenzen aus dem
Nanopore-Datensatz benutzt (90-174 eV). Die ueberlappen mit diesen Daten
(ab 138 eV) nur zu etwa einem Drittel.

Zu diesem Datensatz gehoeren eigene Referenzen: der Ordner `Ca Standards` mit
CaCO3, CaO, Ca(OH)2, CaF2, CaSO4 und metallischem Ca. Damit laesst sich aus der
Feinstruktur der Ca-L2,3-Kante ablesen, in welcher chemischen Form das Calcium
vorliegt.

In [ ]:
# Fenster um die Ca-L2,3-Kante (346 eV). Der Untergrund wird zwischen
# C-K (284 eV) und Ca (346 eV) angepasst.
signal_binned = signal.rebin(scale=[2, 2, 1])
signal_binned = signal_binned.remove_background(signal_range=(300.0, 340.0))
signal_binned = signal_binned.isig[340.0:400.0]

s_smooth = signal_binned.deepcopy()
s_smooth.data = gaussian_filter1d(s_smooth.data, sigma=2, axis=-1)
s_smooth.plot()

In [ ]:
standards = load_standards("ca_standards", sigma=2)

# Die Referenzen sind Rohspektren. Sie muessen genauso vorbehandelt werden wie
# die Messdaten - sonst fittet man untergrundbehaftete Kurven gegen
# untergrundfreie Daten.
for name, s in list(standards.items()):
    s.set_signal_type("EELS")
    s = s.remove_background(signal_range=(300.0, 340.0))
    s = s.isig[340.0:400.0]
    s.data = s.data / s.data.max()
    standards[name] = s
    print(f"{name:16s} {s.axes_manager[-1].size} Kanaele")

In [ ]:
# auto_add_edges=False: die sechs Referenzen beschreiben die Ca-Kante bereits.
# Zusaetzliche Ca-Kantenkomponenten wuerden dasselbe noch einmal modellieren
# und den Fit mehrdeutig machen.
m = s_smooth.create_model(auto_background=False, auto_add_edges=False)

for name, s in standards.items():
    muster = hs.model.components1D.ScalableFixedPattern(s)
    muster.name = name
    muster.xscale.free = False
    muster.shift.free = False
    muster.yscale.bmin = 0
    muster.yscale.bmax = 1e7
    m.append(muster)

m.components

In [ ]:
# --- Variante A: interaktiv ---
m.gui()

In [ ]:
# --- Variante B: per Code ---
# Das Modell enthaelt zweierlei: die Kantenkomponenten (EELSCLEdge) aus
# add_elements und unsere Referenzmuster. Nur letztere haben ein yscale.
for komponente in m:
    yscale = getattr(komponente, "yscale", None)
    if yscale is None:
        print(f"{komponente.name:20s} (Kantenmodell, kein yscale)")
    else:
        print(f"{komponente.name:20s} yscale={yscale.value}")

In [ ]:
m.plot()

In [ ]:
m.multifit(bounded=True)

In [ ]:
m.plot_results()

## Aufgaben

1. Setze `rebin` auf `[4, 4, 1]`. Wie viel laenger dauert der Fit, und siehst du
   in den Elementkarten wirklich mehr?
2. Nimm einzelne Elemente aus `add_elements` heraus. Bei welchen wird der Fit
   sichtbar schlechter, bei welchen aendert sich fast nichts - und was sagt dir das?
3. Schau dir an, welche der sechs Ca-Referenzen den groessten Anteil bekommt.
   Passt das zu dem, was du ueber die Probe weisst?